In [1]:
from datetime import datetime, timedelta

import numpy as np
from scipy.signal import get_window
from stonesoup.models.transition.linear import (
    CombinedLinearGaussianTransitionModel,
    ConstantVelocity,
    KnownTurnRate,
)
from stonesoup.types.array import StateVector
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState
from stonesoup.types.state import State

from nereus.detector import CFARDetector, PassiveSonarDetector, PeakDetector
from nereus.models.environment import FlatBathymetry, Linear
from nereus.models.propagation import (
    rtrsAcousticPropagationModel,
)
from nereus.platform import TowedArrayPlatform
from nereus.signal.ambient import ColouredNoise
from nereus.signal.anthropogenic import BroadbandShipSignal
from nereus.sigproc import (
    DelayAndSumBeamformer,
    MinimumVarianceDistortionlessResponseBeamformer,
    SteeringCalculator,
)
from nereus.simulator import BroadbandPassiveSonarArraySimulator

seed = 12
np.random.seed(seed)


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


# SIMULATION PARAMETERS

In [2]:
# ============================================================================
# SIMULATION PARAMETERS
# ============================================================================

SIM_LENGTH = 900 # seconds
SIM_RATE = 5.0 # seconds

Xlim = [-15_000, 15_000]
Ylim = [-15_000, 15_000]

NUM_TARGETS = 3

SIM_PARAMS = {
    "start_time":datetime.now().replace(hour=0, minute=0, second=0, microsecond=0),
    "time_interval":timedelta(seconds=SIM_RATE),
    "num_steps":int(SIM_LENGTH / SIM_RATE),
}

total_duration_s = SIM_PARAMS["num_steps"] * SIM_PARAMS["time_interval"].total_seconds()
print(f"Total simulation duration: {total_duration_s} seconds")
print(f"Number of timesteps: {SIM_PARAMS['num_steps']}, "
      f"Timestep interval: {SIM_PARAMS['time_interval'].total_seconds()} seconds")

SHIP_PARAMS = {
    "position_mapping": [0, 2, 4],
    "velocity_mapping": [1, 3, 5],
    "transition_model": CombinedLinearGaussianTransitionModel(
        [ConstantVelocity(0), ConstantVelocity(0), ConstantVelocity(0)]
    ),
}

ARRAY_PARAMS = {
    "num_sensors": 200,
    "tow_cable_length": 100.0,
    "sensor_spacing": 0.5,
    "array_depth": -50.0,
}

TARGETS = []

target_sv = [
    StateVector(
        [-1.45627511e+04,
         8.79368717e+00,
         1.20214456e+04,
         -9.79815002e+00,
         -5.00000000e+00,
         0.00000000e+00]
    ),
    StateVector(
        [-1.09436946e+04, -8.05826663e+00, -5.70307247e+03,  3.60050555e+00,
 -5.00000000e+00,  0.00000000e+00]
    ),
    StateVector(
        [-2.98105118e+03,  1.03979014e+01, -9.67307472e+03,  9.71453496e+00,
 -5.00000000e+00,  0.00000000e+00]
    )
]

for i in range(NUM_TARGETS):
    TARGET_PARAMS = {
        "start_vector": target_sv[i],
        "position_mapping": [0, 2, 4],
        "velocity_mapping": [1, 3, 5],
        "transition_model": CombinedLinearGaussianTransitionModel(
            [ConstantVelocity(0), ConstantVelocity(0), ConstantVelocity(0)]
        ),
        "amplitudes_upa": 10 ** (np.random.uniform(87, 102, 4) / 20),
        "frequencies_hz": np.random.uniform(25.0, 200.0, 4),
        "phases_rad": np.random.uniform(0, 2 * np.pi, 4),
        "tonal_bandwidth_hz": np.random.uniform(0.5, 2.0),
        "noise_amplitude_upa":10 ** (np.random.uniform(65, 85) / 20),
        "noise_spectral_exponent":-1.0,
    }
    TARGETS.append(TARGET_PARAMS)

SIGNAL_PARAMS = {
    "duration_s": total_duration_s,
    "sampling_rate_hz": 500.0,
    "frame_len": 500,
    "hop_factor": 2,
    "fade_in_ms": 1000.0,
}

AMBIENT_NOISE_PARAMS = {
    "amplitude_upa": 10 ** (np.random.uniform(45, 55) / 20),
    "spectral_exponent": -1,
}

PROP_PARAMS = {
    "ssp": Linear(surface_speed=1500.0, gradient=0.2),
    "attenuation_factor": 0.5,
    "bathymetry": FlatBathymetry(depth=-150.0),
    "step_m": 20.0,
    "azimuth_search_width": 2.0,
    "azimuth_resolution": 0.5,
    "elevation_range": (-25.0, 25.0),
    "elevation_resolution": 1.0,
}

BF_PARAMS = {
    "beamformer_type": "DAS",
    "shading": None,
    "domain": "frequency",
    "steering_azimuths_rad": np.linspace(-np.pi, np.pi, 361),
}

# Detection parameters
DET_PARAMS = {
    "cfar_detector": {
        "num_guard_cells": 6,
        "num_training_cells": 10,
        "threshold_factor": 1.05,
        "mode": "wrap"
    },
    "peak_detector": {
        "distance": 3,
    }
}

Total simulation duration: 900.0 seconds
Number of timesteps: 180, Timestep interval: 5.0 seconds


# SETUP

In [3]:
# ============================================================================
# SETUP
# ============================================================================
platform_turn_rate_radps = np.deg2rad(1.0)

leg1_duration_s = timedelta(seconds=405)

turn1_angle_rad = np.deg2rad(-45)
turn1_duration_s = timedelta(seconds=round(
    (abs(turn1_angle_rad) / platform_turn_rate_radps) / SIM_RATE)
    * SIM_RATE
)

leg2_duration_s = timedelta(seconds=SIM_LENGTH) - leg1_duration_s - turn1_duration_s

print(f"Leg 1 duration: {leg1_duration_s.total_seconds()} seconds")
print(f"Turn 1 duration: {turn1_duration_s.total_seconds()} seconds")
print(f"Leg 2 duration: {leg2_duration_s.total_seconds()} seconds")

straight_model = CombinedLinearGaussianTransitionModel([
    ConstantVelocity(0.0),
    ConstantVelocity(0.0),
    ConstantVelocity(0.0)
])

turn_rate_rad1 = np.sign(turn1_angle_rad) * platform_turn_rate_radps
planar_turn1 = KnownTurnRate(
    turn_rate=turn_rate_rad1,
    turn_noise_diff_coeffs=np.array([0.0, 0.0])
)
depth_model = ConstantVelocity(0.0)
turning_model1 = CombinedLinearGaussianTransitionModel([
    planar_turn1,  # Handles indices 0, 1, 2, 3 (x, vx, y, vy)
    depth_model   # Handles indices 4, 5 (z, vz)
])

plat_init_sv = StateVector([-7500,
                            1.92039757e+00,
                            -2000,
                            2.69915147e-01,
                            -5.00000000e+00,
                            0.00000000e+00])

platform_initial_state = State(
    plat_init_sv,
    timestamp=SIM_PARAMS["start_time"]
)

transition_models = [
    straight_model, turning_model1, straight_model
]
transition_times = [
    leg1_duration_s, turn1_duration_s, leg2_duration_s
]

# Create platform
print("Creating platform...")

platform = TowedArrayPlatform(
    states=platform_initial_state,
    position_mapping=SHIP_PARAMS["position_mapping"],
    velocity_mapping=SHIP_PARAMS["velocity_mapping"],
    transition_models=transition_models,
    transition_times=transition_times,
    num_sensors=ARRAY_PARAMS["num_sensors"],
    cable_length_m=ARRAY_PARAMS["tow_cable_length"],
    sensor_spacing_m=ARRAY_PARAMS["sensor_spacing"],
    array_depth_m=ARRAY_PARAMS["array_depth"],
)

# Move platform through all timesteps with maneuver transitions
print("Moving platform through scenario...")

for i in range(1, SIM_PARAMS["num_steps"]):
    new_time = SIM_PARAMS["start_time"] + i * SIM_PARAMS["time_interval"]
    platform.move(new_time)

# Create target trajectories
print(f"Creating {NUM_TARGETS} target trajectories with random maneuvers...")
target_ground_truths = []
relative_bearing_ground_truths = []

for _, TARGET_PARAMS in enumerate(TARGETS):

    # Create initial state
    target_states = [
        GroundTruthState(
            TARGET_PARAMS["start_vector"],
            timestamp=SIM_PARAMS["start_time"],
            metadata={
                "amplitudes_upa": TARGET_PARAMS["amplitudes_upa"],
                "frequencies_hz": TARGET_PARAMS["frequencies_hz"],
                "phases_rad": TARGET_PARAMS["phases_rad"],
                "position_mapping": TARGET_PARAMS["position_mapping"],
                "velocity_mapping": TARGET_PARAMS["velocity_mapping"],
                "tonal_bandwidth_hz": TARGET_PARAMS["tonal_bandwidth_hz"],
                "noise_amplitude_upa": TARGET_PARAMS["noise_amplitude_upa"],
                "noise_spectral_exponent": TARGET_PARAMS["noise_spectral_exponent"],
            },
        )
    ]

    # Propagate trajectory with multi-transition models
    current_maneuver_idx = 0

    for i in range(1, SIM_PARAMS["num_steps"]):
        transition_model = TARGET_PARAMS["transition_model"]
        new_time = SIM_PARAMS["start_time"] + i * SIM_PARAMS["time_interval"]
        time_interval = new_time - target_states[-1].timestamp
        new_state_vector = transition_model.function(
            target_states[-1], noise=False, time_interval=time_interval
        )
        new_state = GroundTruthState(
            new_state_vector,
            timestamp=new_time,
            metadata=target_states[-1].metadata,
        )
        target_states.append(new_state)

    target_ground_truth = GroundTruthPath(target_states)
    target_ground_truths.append(target_ground_truth)

    # Compute ground truth bearings for plotting
    gt_relative_bearings = []
    for target_state in target_ground_truth.states:
        platform_state = platform.get_platform_state_at(target_state.timestamp)
        ref_sensor_position = np.mean(platform_state.array.state_vector, axis=1)
        target_pos = np.array(
            [target_state.state_vector[0], target_state.state_vector[2]]
        )
        relative_pos = target_pos - ref_sensor_position[:2]
        bearing_rad = np.arctan2(relative_pos[1], relative_pos[0])
        gt_relative_bearings.append(bearing_rad)

    gt_relative_bearings = np.array(gt_relative_bearings)

    # Create GroundTruthPath for bearings
    relative_bearing_truth_states = []
    for i, bearing in enumerate(gt_relative_bearings):
        timestamp = SIM_PARAMS["start_time"] + i * SIM_PARAMS["time_interval"]
        bearing_state = GroundTruthState(
            state_vector=np.array([bearing]),
            timestamp=timestamp
        )
        relative_bearing_truth_states.append(bearing_state)

    relative_bearing_ground_truth = GroundTruthPath(relative_bearing_truth_states)
    relative_bearing_ground_truths.append(relative_bearing_ground_truth)

# Create signal and propagation models
print("Creating signal and propagation models...")

prop_model = rtrsAcousticPropagationModel(
    ssp=PROP_PARAMS["ssp"],
    bathymetry=PROP_PARAMS["bathymetry"],
    use_all_frequencies=False,  # Will use propagate_spectrum
    step_m=20.0,
    azimuth_search_width=2.0,
    azimuth_resolution=0.5,
    elevation_range=(-25.0, 25.0),
    elevation_resolution=1.0,
)

# Create ambient noise model
ambient_noise_model = ColouredNoise(
    amplitude_upa=AMBIENT_NOISE_PARAMS["amplitude_upa"],
    spectral_exponent=AMBIENT_NOISE_PARAMS["spectral_exponent"],
    duration_s=SIM_PARAMS["time_interval"].total_seconds(),
    sampling_rate_hz=SIGNAL_PARAMS["sampling_rate_hz"],
)

# Create beamformer and steering calculator
print("Creating beamformer and steering calculator...")


shading = None
if BF_PARAMS["shading"] is not None:
    shading = get_window(BF_PARAMS["shading"], platform.num_sensors)

if BF_PARAMS["beamformer_type"] == "DAS":
    if BF_PARAMS["domain"] == "broadband_power":
        beamformer = DelayAndSumBeamformer(
            domain=BF_PARAMS["domain"],
            sampling_rate_hz=SIGNAL_PARAMS["sampling_rate_hz"],
            fmin=BF_PARAMS["fmin"],
            fmax=BF_PARAMS["fmax"],
        )
    else:
        beamformer = DelayAndSumBeamformer(
            sampling_rate_hz=SIGNAL_PARAMS["sampling_rate_hz"],
            shading=shading,
            domain=BF_PARAMS["domain"]
        )
elif BF_PARAMS["beamformer_type"] == "MVDR":

    beamformer = MinimumVarianceDistortionlessResponseBeamformer(
        sampling_rate_hz=SIGNAL_PARAMS["sampling_rate_hz"],
        fmin=BF_PARAMS["fmin"],
        fmax=BF_PARAMS["fmax"],
    )
else:
    raise ValueError(f"Unknown beamformer type: {BF_PARAMS['beamformer_type']}")


steering_calculator = SteeringCalculator(
    ssp=PROP_PARAMS["ssp"],
    steering_azimuths_rad=BF_PARAMS["steering_azimuths_rad"],
)

# Create signal models for each target
print("Creating signal models for each target...")
signal_models = []
for _, TARGET_PARAMS in enumerate(TARGETS):
    signal_model = BroadbandShipSignal(
        duration_s=SIGNAL_PARAMS["duration_s"],
        sampling_rate_hz=SIGNAL_PARAMS["sampling_rate_hz"],
        frame_len=SIGNAL_PARAMS["frame_len"],
        hop_factor=SIGNAL_PARAMS["hop_factor"],
        tonal_bandwidth_hz=TARGET_PARAMS["tonal_bandwidth_hz"],
        noise_amplitude_upa=TARGET_PARAMS["noise_amplitude_upa"],
        noise_spectral_exponent=TARGET_PARAMS["noise_spectral_exponent"],
        noise_freq_range_hz=(0.0, SIGNAL_PARAMS["sampling_rate_hz"]/2),
        tonal_noise_is_constant=True,
        noise_is_constant=True,
    )
    signal_models.append(signal_model)

# Use the first signal model for simplicity (have same time/freq params but different
# source signal)
signal_model = signal_models[0]

# Create simulator with beamforming
print("Creating broadband simulator with beamforming...")
simulator = BroadbandPassiveSonarArraySimulator(
    platform=platform,
    propagation_model=prop_model,
    signal_model=signal_model,
    noise_model=ambient_noise_model,
    beamformer=beamformer,
    steering_calculator=steering_calculator,
    ground_truth_paths=target_ground_truths,
    fade_in_ms=SIGNAL_PARAMS["fade_in_ms"],
)

for target_idx, TARGET_PARAMS in enumerate(TARGETS):
    print(f"  Target {target_idx + 1}:")
    print(f"    Frequencies: {TARGET_PARAMS['frequencies_hz']} Hz")
    print(f"    Amplitudes: {20 * np.log10(TARGET_PARAMS['amplitudes_upa'])} "
          "dB re 1 µPa")
    print(f"    Start Position: ({TARGET_PARAMS['start_vector'][0]:.0f}, "
          f"{TARGET_PARAMS['start_vector'][2]:.0f}) m")
    print(f"    Velocity: ({TARGET_PARAMS['start_vector'][1]:.1f}, "
          f"{TARGET_PARAMS['start_vector'][3]:.1f}) m/s")
    print(f"    Bandwidth: {TARGET_PARAMS['tonal_bandwidth_hz']} Hz")
    print(f"    Noise Amplitude: {20 * np.log10(TARGET_PARAMS['noise_amplitude_upa'])} "
          "dB re 1 µPa")

Leg 1 duration: 405.0 seconds
Turn 1 duration: 45.0 seconds
Leg 2 duration: 450.0 seconds
Creating platform...
Moving platform through scenario...
Creating 3 target trajectories with random maneuvers...
Creating signal and propagation models...
Creating beamformer and steering calculator...
Creating signal models for each target...
Creating broadband simulator with beamforming...
  Target 1:
    Frequencies: [ 27.55061843 185.78072642 182.62509947  30.84874983] Hz
    Amplitudes: [89.31244264 98.10074545 90.94972523 95.0060909 ] dB re 1 µPa
    Start Position: (-14563, 12021) m
    Velocity: (8.8, -9.8) m/s
    Bandwidth: 1.9163377040795633 Hz
    Noise Amplitude: 82.05471082218568 dB re 1 µPa
  Target 2:
    Frequencies: [159.42347696  53.1254318  158.79807881  28.64171464] Hz
    Amplitudes: [87.0338885  94.81839041 95.2805645  94.2806612 ] dB re 1 µPa
    Start Position: (-10944, -5703) m
    Velocity: (-8.1, 3.6) m/s
    Bandwidth: 1.2068446673750213 Hz
    Noise Amplitude: 81.3233

In [ ]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# --- 1. Prepare Data (Convert to km) ---
timesteps = [
    SIM_PARAMS["start_time"] + i * SIM_PARAMS["time_interval"]
    for i in range(SIM_PARAMS["num_steps"])
]

plat_x = []
plat_y = []
for entry in platform.platform_history:
    plat_x.append(entry.host.state.state_vector[0] / 1000)
    plat_y.append(entry.host.state.state_vector[2] / 1000)

tgt_x = [[] for _ in range(NUM_TARGETS)]
tgt_y = [[] for _ in range(NUM_TARGETS)]
for idx, target_ground_truth in enumerate(target_ground_truths):
    tgt_x[idx] = [state.state_vector[0] / 1000 for state in target_ground_truth]
    tgt_y[idx] = [state.state_vector[2] / 1000 for state in target_ground_truth]

fig = go.Figure()

# --- 2. Add Traces ---

# A. Main Plot Traces
fig.add_trace(go.Scatter(
    x=plat_x, y=plat_y,
    mode='lines', line=dict(color='black', width=3),
    name='Platform'
))

colors = px.colors.qualitative.Dark24
names = [f"Target {i+1}" for i in range(NUM_TARGETS)]

for i in range(NUM_TARGETS):
    fig.add_trace(go.Scatter(
        x=tgt_x[i], y=tgt_y[i],
        mode='lines', line=dict(color=colors[i], width=3, dash="5px,2px"),
        name=names[i]
    ))

# B. Inset Trace (Platform Only)
fig.add_trace(go.Scatter(
    x=plat_x, y=plat_y,
    mode='lines+markers',
    marker=dict(size=3, color='black'),
    line=dict(color='black', width=1.5),
    showlegend=False,
    xaxis='x2', yaxis='y2'
))

# --- 3. Calculate Ranges & Force Square Geometry ---

# A. Square the Main Axes
# Gather all data points to find the extent
all_x = plat_x + [x for sublist in tgt_x for x in sublist]
all_y = plat_y + [y for sublist in tgt_y for y in sublist]

# Find min/max with a small buffer
min_x, max_x = min(all_x) - 0.5, max(all_x) + 0.5
min_y, max_y = min(all_y) - 0.5, max(all_y) + 0.5

# Calculate the maximum span to force a 1:1 ratio
mid_x = (max_x + min_x) / 2
mid_y = (max_y + min_y) / 2
max_span = max(max_x - min_x, max_y - min_y)

main_x_range = [mid_x - max_span/2, mid_x + max_span/2]
main_y_range = [mid_y - max_span/2, mid_y + max_span/2]

# B. Square the Zoom Box (Platform Only)
p_min_x, p_max_x = min(plat_x), max(plat_x)
p_min_y, p_max_y = min(plat_y), max(plat_y)
p_mid_x = (p_min_x + p_max_x) / 2
p_mid_y = (p_min_y + p_max_y) / 2

# Force the zoom box to be square as well
zoom_pad = 0.2
zoom_span = max(p_max_x - p_min_x, p_max_y - p_min_y) + (zoom_pad * 2)

box_x = [p_mid_x - zoom_span/2, p_mid_x + zoom_span/2]
box_y = [p_mid_y - zoom_span/2, p_mid_y + zoom_span/2]

# C. Define Inset Position (Domain 0-1)
inset_dom_x = [0.6, 1.0]
inset_dom_y = [0.6, 1.0]

# --- 4. Configure Layout ---
fig.update_layout(
    width=600, height=600,
    font=dict(family="Times New Roman", size=16, color="black"),
    showlegend=True,
    legend=dict(x=0.5, y=-0.15, xanchor="center", orientation="h"),
    plot_bgcolor="white",

    # Main Axes (Squared)
    xaxis=dict(title="X Position (km)", range=main_x_range, showgrid=True,
               gridcolor="rgba(200,200,200,0.5)", linecolor="black", zeroline=True,
               zerolinecolor="rgba(200, 200, 200, 0.5)", zerolinewidth=0.5),
    yaxis=dict(title="Y Position (km)", range=main_y_range, showgrid=True,
               gridcolor="rgba(200,200,200,0.5)", linecolor="black", zeroline=True,
               zerolinecolor="rgba(200, 200, 200, 0.5)", zerolinewidth=0.5),

    # Inset Axes (Squared Box + Domain)
    xaxis2=dict(
        domain=inset_dom_x, anchor='y2', range=box_x,
        showgrid=True, gridcolor="rgba(200,200,200,0.25)",
        linecolor="rgba(0,0,0,0.5)", tickfont=dict(size=10)
    ),
    yaxis2=dict(
        domain=inset_dom_y, anchor='x2', range=box_y,
        showgrid=True, gridcolor="rgba(200,200,200,0.25)",
        linecolor="rgba(0,0,0,0.5)", tickfont=dict(size=10)
    ),
)

# --- 5. Add Shapes (Background, Box, Lines) ---

# A. White Background for Inset (Using DOMAIN coordinates)
# Placing it on "domain" layer allows the axis grid to render on top of it.
fig.add_shape(
    type="rect",
    xref="x domain", yref="y domain",
    x0=inset_dom_x[0], y0=inset_dom_y[0],
    x1=inset_dom_x[1], y1=inset_dom_y[1],
    fillcolor="white",
    line=dict(width=0),
    layer="below"
)

# B. Zoom Box around Platform (Using DATA coordinates)
fig.add_shape(
    type="rect",
    xref="x", yref="y",
    x0=box_x[0], y0=box_y[0],
    x1=box_x[1], y1=box_y[1],
    line=dict(color="rgba(0,0,0,0.5)", width=1),
    fillcolor="rgba(0,0,0,0)"
)

# C. Connecting Lines (Data -> Data Calculation)
# Convert Domain edges to Data coordinates for the connection points
def domain_to_data(dom_val, r_min, r_max):
    """Convert domain value (0-1) to data coordinate based on axis range."""
    return r_min + (dom_val * (r_max - r_min))

inset_left_km   = domain_to_data(inset_dom_x[0], main_x_range[0], main_x_range[1])
inset_right_km  = domain_to_data(inset_dom_x[1], main_x_range[0], main_x_range[1])
inset_bottom_km = domain_to_data(inset_dom_y[0], main_y_range[0], main_y_range[1])
inset_top_km    = domain_to_data(inset_dom_y[1], main_y_range[0], main_y_range[1])

# Line 1: Top-Left Box -> Top-Left Inset
fig.add_shape(
    type="line",
    xref="x", yref="y",
    x0=box_x[0],      y0=box_y[1],     # Box Corner
    x1=inset_left_km, y1=inset_top_km, # Inset Corner
    line=dict(color="rgba(0,0,0,0.5)", width=1, dash="solid"),
    layer="above"
)

# Line 2: Bottom-Right Box -> Bottom-Right Inset
fig.add_shape(
    type="line",
    xref="x", yref="y",
    x0=box_x[1],       y0=box_y[0],        # Box Corner
    x1=inset_right_km, y1=inset_bottom_km, # Inset Corner
    line=dict(color="rgba(0,0,0,0.5)", width=1, dash="solid"),
    layer="above"
)

# --- 6. Add Directional Arrows ---
def add_arrowhead(fig, x, y, i, color, size, xref="x", yref="y"):
    """Add a directional arrow shape."""
    if i < 1 or i >= len(x):
        return
    dx = x[i] - x[i-1]
    dy = y[i] - y[i-1]
    magnitude = (dx**2 + dy**2)**0.5
    if magnitude > 0:
        ux, uy = dx/magnitude, dy/magnitude
        vx, vy = -uy, ux
        tip_x, tip_y = x[i], y[i]
        base_x = tip_x - (ux * size)
        base_y = tip_y - (uy * size)
        hw = size * 0.35
        path = (f"M {tip_x},{tip_y} L {base_x + vx*hw},{base_y + vy*hw} L "
                f"{base_x - vx*hw},{base_y - vy*hw} Z")
        fig.add_shape(type="path", path=path, fillcolor=color,
                      line=dict(color=color, width=1), xref=xref, yref=yref,
                      layer="above")

# Main Plot Arrows
for tgt_idx in range(NUM_TARGETS):
    for i in range(40, len(tgt_x[tgt_idx]), 30):
        add_arrowhead(fig, tgt_x[tgt_idx], tgt_y[tgt_idx], i, colors[tgt_idx], size=0.7)

# Inset Arrows
for i in range(40, len(plat_x), 40):
    add_arrowhead(fig, plat_x, plat_y, i, "black", size=0.15, xref="x2", yref="y2")

fig.show()
fig.write_image("figs/mt_world_picture.pdf")

# RUN SIMULATION

In [5]:
# ============================================================================
# RUN SIMULATION WITH DETECTION
# ============================================================================

data_generator = simulator.sensor_data_gen()

# Create detection chain
cfar_detector = CFARDetector(
    num_guard_cells=DET_PARAMS["cfar_detector"]["num_guard_cells"],
    num_training_cells=DET_PARAMS["cfar_detector"]["num_training_cells"],
    threshold_factor=DET_PARAMS["cfar_detector"]["threshold_factor"],
    mode=DET_PARAMS["cfar_detector"]["mode"]
)
peak_detector = PeakDetector(distance=DET_PARAMS["peak_detector"]["distance"])

detection_chain = [cfar_detector]
detection_chain.append(peak_detector)

detector = PassiveSonarDetector(
    detection_chain=detection_chain,
    sensor_data_gen=data_generator,
    steering_azimuths_rad=BF_PARAMS["steering_azimuths_rad"]
)

# Run detection
print("Running detection chain...")
all_detections = list(detector.detections_gen(progress_bar=True))
snr_map = detector.snr_history

Running detection chain...


Generating Detections: 165it [03:18,  1.21s/it]


KeyboardInterrupt: 

# TARGET TRACKING

In [ ]:
import numpy as np
from stonesoup.dataassociator.neighbour import GNNWith2DAssignment
from stonesoup.dataassociator.probability import JPDA
from stonesoup.deleter.error import CovarianceBasedDeleter
from stonesoup.functions import gm_reduce_single, mod_bearing
from stonesoup.hypothesiser.distance import DistanceHypothesiser
from stonesoup.hypothesiser.probability import PDAHypothesiser
from stonesoup.initiator.simple import MultiMeasurementInitiator
from stonesoup.measures import Mahalanobis
from stonesoup.models.measurement.linear import LinearGaussian
from stonesoup.models.transition.linear import ConstantVelocity
from stonesoup.predictor.kalman import KalmanPredictor
from stonesoup.types.angle import Bearing
from stonesoup.types.array import StateVectors
from stonesoup.types.detection import MissedDetection
from stonesoup.types.state import GaussianState
from stonesoup.types.update import GaussianStateUpdate
from stonesoup.updater.kalman import ExtendedKalmanUpdater

# ============================================================================
# Models
# ============================================================================
transition_model = ConstantVelocity(0.000001)
predictor = KalmanPredictor(transition_model)

measurement_model = LinearGaussian(
    ndim_state=2,
    mapping=[0],
    noise_covar=np.array([[np.deg2rad(1)**2]])
)
updater = ExtendedKalmanUpdater(measurement_model=measurement_model)

# ============================================================================
# JPDA for main tracking
# ============================================================================
FOV_RAD = np.deg2rad(360)
expected_false_alarms_per_scan = 6
clutter_spatial_density = expected_false_alarms_per_scan / FOV_RAD

jpda_hypothesiser = PDAHypothesiser(
    predictor=predictor,
    updater=updater,
    clutter_spatial_density=clutter_spatial_density,
    prob_detect=0.95
)
jpda_associator = JPDA(hypothesiser=jpda_hypothesiser)

# ============================================================================
# GNN for initiation (single-hypothesis, avoids MultipleHypothesis issue)
# ============================================================================
init_hypothesiser = DistanceHypothesiser(
    predictor=predictor,
    updater=updater,
    measure=Mahalanobis(),
    missed_distance=3
)
init_associator = GNNWith2DAssignment(init_hypothesiser)

# ============================================================================
# Track management
# ============================================================================
deleter = CovarianceBasedDeleter(covar_trace_thresh=0.1)

relative_bearing_ground_truth = relative_bearing_ground_truths[0]
initial_bearing = float(relative_bearing_ground_truth[0].state_vector[0]) + \
      np.random.normal(0, np.deg2rad(2))
initial_bearing = mod_bearing(initial_bearing)

prior_state = GaussianState(
    np.array([[initial_bearing], [0.0]]),
    np.diag([np.deg2rad(5)**2, np.deg2rad(0.5)**2]),
    timestamp=SIM_PARAMS["start_time"]
)
Bearing(prior_state.state_vector[0,0])

initiator = MultiMeasurementInitiator(
    prior_state=prior_state,
    measurement_model=measurement_model,
    deleter=deleter,
    data_associator=init_associator,
    updater=updater,
    min_points=30,
)

tracks, all_tracks = set(), set()


# ============================================================================
# Main loop
# ============================================================================
for timestamp, detections in all_detections:

    # Wrap detections
    for det in detections:
        det.state_vector[0, 0] = mod_bearing(float(det.state_vector[0, 0]))

    # JPDA association for existing tracks
    associations = jpda_associator.associate(set(tracks), detections, timestamp)

    associated_detections = set()

    for track in list(tracks):
        track_hypotheses = associations[track]  # MultipleHypothesis

        posterior_states = []
        posterior_weights = []

        for hyp in track_hypotheses:
            if hyp.measurement is None or isinstance(hyp.measurement, MissedDetection):
                state = hyp.prediction
                Bearing(state.state_vector[0,0])
            else:
                state = updater.update(hyp)
                Bearing(state.state_vector[0,0])
                associated_detections.add(hyp.measurement)

            posterior_states.append(state)
            posterior_weights.append(float(hyp.probability))

        means = StateVectors([s.state_vector for s in posterior_states])
        covars = np.stack([s.covar for s in posterior_states], axis=2)
        weights = np.asarray(posterior_weights, dtype=float)
        if weights.sum() > 0:
            weights /= weights.sum()

        post_mean, post_covar = gm_reduce_single(means, covars, weights)
        post_mean = post_mean.copy()
        post_mean[0, 0] = mod_bearing(float(post_mean[0, 0]))

        track.append(
            GaussianStateUpdate(post_mean, post_covar, track_hypotheses, timestamp)
        )

    # Delete and initiate (initiation uses GNN associator)
    tracks -= deleter.delete_tracks(tracks)
    tracks |= initiator.initiate(detections - associated_detections, timestamp)
    all_tracks |= tracks

In [ ]:
import plotly.express as px
from plotly.subplots import make_subplots

det_x = []
det_y = []

for _, detections in all_detections:
    for det in detections:
        det_x.append(np.rad2deg(det.state_vector[0]))
        det_y.append(det.timestamp)

track_x = [
    [np.rad2deg(state.state_vector[0]) for state in track]
    for track in all_tracks]
track_y = [
    [state.timestamp for state in track]
    for track in all_tracks]

gt_x = [
    [np.rad2deg(state.state_vector[0]) for state in relative_bearing_ground_truth]
    for relative_bearing_ground_truth in relative_bearing_ground_truths]
gt_y = [
    [state.timestamp for state in relative_bearing_ground_truth]
    for relative_bearing_ground_truth in relative_bearing_ground_truths]

fig = make_subplots(
    rows=1, cols=3, shared_xaxes=True, shared_yaxes=True, horizontal_spacing=0.06,
    subplot_titles=["(a)", "(b)", "(c)"]
)

fig.add_trace(
    go.Heatmap(
        z=snr_map,
        y=timesteps,
        x=np.rad2deg(BF_PARAMS["steering_azimuths_rad"]),
        colorscale="Viridis",
        colorbar=dict(
            title=dict(text="SNR (dB)", side="right", font=dict(size=16)),
            thickness=24,
            len=1.0,
            tickfont=dict(size=14),
        ),
    ),
    row=1, col=1
)

fig.add_trace(
    go.Heatmap(
        z=snr_map,
        y=timesteps,
        x=np.rad2deg(BF_PARAMS["steering_azimuths_rad"]),
        colorscale="Viridis",
        showscale=False
    ),
    row=1, col=2
)

fig.add_trace(
    go.Scatter(
        x=det_x,
        y=det_y,
        mode="markers",
        name="Detection",
        marker=dict(
            size=6,
            line=dict(width=1),
            color="white",
            opacity=1.0,
        ),
        hovertemplate="Bearing: %{x:.1f}°<br>Time: %{y|%H:%M:%S}<extra></extra>",
    ),
    row=1, col=2
)

fig.add_trace(
    go.Scatter(
        x=det_x,
        y=det_y,
        mode="markers",
        name="Detection",
        showlegend=False,
        marker=dict(
            size=4,
            line=dict(width=1),
            color="white",
            opacity=0.5,
        ),
        hovertemplate="Bearing: %{x:.1f}°<br>Time: %{y|%H:%M:%S}<extra></extra>",
    ),
    row=1, col=3
)

def handle_wraparound(x_vals, y_vals, threshold=180):
    """Detect wraparound jumps and insert None to break the Plotly line."""
    if not x_vals or not y_vals:
        return [], []

    clean_x = [x_vals[0]]
    clean_y = [y_vals[0]]

    for i in range(1, len(x_vals)):
        # If the jump is huge (e.g., 179 to -179 is a diff of 358), it's a wrap
        if abs(x_vals[i] - x_vals[i-1]) > threshold:
            clean_x.append(None) # Breaks the line in Plotly
            clean_y.append(None)

        clean_x.append(x_vals[i])
        clean_y.append(y_vals[i])

    return clean_x, clean_y

print(len(all_tracks))

# Cool Colors: Blues, Greens, Purples, Cyans
# derived from px.colors.qualitative.Plotly and D3
track_colors = [
    '#1f77b4',  # Muted Blue
    '#2ca02c',  # Cooked Asparagus Green
    '#9467bd',  # Muted Purple
    '#17becf',  # Blue-Teal
    '#000080',  # Navy Blue
    '#006400',  # Dark Green
    '#4b0082',  # Indigo
]

# Warm Colors: Reds, Oranges, Pinks, Yellows
truth_colors = [
    '#d62728',  # Brick Red
    '#ff7f0e',  # Safety Orange
    '#e377c2',  # Raspberry Yogurt Pink
    '#bcbd22',  # Curry Yellow-Green
    '#8c564b',  # Chestnut Brown
    '#ff0000',  # Pure Red
    '#ff1493',  # Deep Pink
]

# colours = px.colors.qualitative.Dark24
for i in range(len(all_tracks)):
    wx, wy = handle_wraparound(track_x[i], track_y[i])
    show_legend = (i == 0)
    fig.add_trace(
        go.Scatter(
            x=wx,
            y=wy,
            mode="lines",
            name="Track",
            line=dict(color=track_colors[i % len(track_colors)], width=4),
            showlegend=show_legend
        ),
        row=1, col=3
    )

# colours = px.colors.qualitative.Light24
for i in range(len(relative_bearing_ground_truths)):
    wx, wy = handle_wraparound(gt_x[i], gt_y[i])
    show_legend = (i == 0)
    fig.add_trace(
        go.Scatter(
            x=wx,
            y=wy,
            mode="lines",
            name="Ground Truth",
            line=dict(color=truth_colors[i % len(truth_colors)], width=3, dash="dash"),
            showlegend=show_legend
        ),
        row=1, col=3
    )

fig.update_xaxes(
    range=[-180, 180],
    tickmode="linear",
    tick0=-180,
    dtick=60,
    tickangle=-45,
    tickfont=dict(size=14),
    showgrid=True,
    gridcolor="rgba(200, 200, 200, 0.5)",
    title="Bearing (°)",
    ticks="outside",
    tickcolor="rgba(160, 160, 160, 1.0)"
)
fig.update_xaxes(title="", col=1)
fig.update_xaxes(
    title="",
    col=3,
    showline=True,
    linewidth=1,
    linecolor="rgba(160, 160, 160, 1.0)"
)

fig.update_yaxes(
    range=[timesteps[-1], timesteps[0]],
    showgrid=True,
    gridcolor="rgba(200, 200, 200, 0.5)",
    tickformat="%H:%M",
    tickfont=dict(size=14),
    autorange=False,
    title="Time (HH:MM)",
    tickcolor="rgba(160, 160, 160, 1.0)"
)
fig.update_yaxes(ticks="outside", col=1)
fig.update_yaxes(showticklabels=False, title="", col=2)
fig.update_yaxes(
    showticklabels=False,
    title="",
    col=3,
    showline=True,
    linewidth=1,
    linecolor="rgba(160, 160, 160, 1.0)"
)

fig.update_layout(
    width=1200,
    height=600,
    margin=dict(b=100),
    font=dict(family="Times New Roman", size=16, color="black"),
    showlegend=True,
    legend=dict(
        x=0.5,
        y=-0.3,
        xanchor="center",
        yanchor="bottom",
        bgcolor="rgba(255,255,255,0.0)",
        borderwidth=0,
        orientation="h"
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
)

fig.show()

fig.write_image("figs/mt_bf_tracker.pdf", scale=1, width=1200, height=600)

# STONE SOUP DETECTIONS

In [ ]:
from scipy.stats import uniform
from stonesoup.types.detection import Detection

# Measurement model for bearing-only measurements
deg_std = 0.5
detection_measurement_model = LinearGaussian(
    ndim_state=1,
    mapping=[0],
    noise_covar=np.array([[np.deg2rad(deg_std)**2]])
)

# Measurement model for tracking (2D state -> 1D measurement)
ss_measurement_model = LinearGaussian(
    ndim_state=2,
    mapping=[0],
    noise_covar=np.array([[np.deg2rad(deg_std)**2]])
)

# Detection parameters
prob_detection = 0.95
FOV_RAD = np.deg2rad(360)
expected_false_alarms_per_scan = 1
clutter_spatial_density = expected_false_alarms_per_scan / FOV_RAD

# Generate detections from ground truth
stone_soup_detections = []

for i, timestamp in enumerate(timesteps):
    detections_at_time = []

    # Generate true detections from each target
    for gt_bearing_path in relative_bearing_ground_truths:
        if np.random.rand() < prob_detection:
            measurement = detection_measurement_model.function(
                gt_bearing_path[i], noise=True
            )

            detection = Detection(
                state_vector=measurement,
                timestamp=timestamp,
                measurement_model=ss_measurement_model
            )
            detections_at_time.append(detection)

    # Add clutter/false alarms
    num_clutter = np.random.poisson(clutter_spatial_density)
    for _ in range(num_clutter):
        clutter_bearing = uniform.rvs(loc=-np.pi, scale=2*np.pi)
        clutter_detection = Detection(
            state_vector=np.array([[clutter_bearing]]),
            timestamp=timestamp,
            measurement_model=ss_measurement_model
        )
        detections_at_time.append(clutter_detection)

    stone_soup_detections.append((timestamp, detections_at_time))

# Convert to format compatible with plotter
ss_detections_for_plotter = [det_set for _, det_set in stone_soup_detections]


In [ ]:
from scipy.stats import uniform
from stonesoup.types.detection import Detection

SS_DET_PARAMS = {
    "prob_detection": 0.95,
    "bearing_std_deg": 0.5,
    "fov_deg": 360.0,
    "expected_clutter_per_scan": 1,
    "include_ambiguity": True,
}

def compute_bearing_world_frame(platform_state, target_position):
    """Compute bearing from platform array center to target position in world frame."""
    array_center = np.mean(platform_state.array.state_vector, axis=1)
    relative_pos = target_position - array_center[:2]
    absolute_bearing = np.arctan2(relative_pos[1], relative_pos[0])
    return np.arctan2(np.sin(absolute_bearing), np.cos(absolute_bearing))

def generate_ambiguous_bearing(true_bearing_rad, platform_heading_rad):
    """Generate port/starboard ambiguous bearing."""
    ambiguous_bearing = 2 * platform_heading_rad - true_bearing_rad
    return np.arctan2(np.sin(ambiguous_bearing), np.cos(ambiguous_bearing))

# Measurement models
deg_std = SS_DET_PARAMS["bearing_std_deg"]
detection_measurement_model = LinearGaussian(
    ndim_state=1,
    mapping=[0],
    noise_covar=np.array([[np.deg2rad(deg_std)**2]])
)

ss_measurement_model = LinearGaussian(
    ndim_state=2,
    mapping=[0],
    noise_covar=np.array([[np.deg2rad(deg_std)**2]])
)

# Detection parameters
prob_detection = SS_DET_PARAMS["prob_detection"]
FOV_RAD = np.deg2rad(SS_DET_PARAMS["fov_deg"])
expected_false_alarms_per_scan = SS_DET_PARAMS["expected_clutter_per_scan"]
clutter_spatial_density = expected_false_alarms_per_scan / FOV_RAD

print("Generating Stone Soup detections with left-right ambiguity...")
stone_soup_detections = []

for i, timestamp in enumerate(timesteps):
    detections_at_time = []

    # Generate true detections from each target
    for target_gt in target_ground_truths:
        target_state = target_gt[i]

        # Compute true bearing
        platform_state = platform.get_platform_state_at(timestamp)
        target_pos = np.array(
            [target_state.state_vector[0], target_state.state_vector[2]]
        )
        true_bearing = compute_bearing_world_frame(platform_state, target_pos)

        # Detection probability
        if np.random.rand() < prob_detection:
            # Generate noisy measurement for true bearing
            true_bearing_state = GroundTruthState(
                state_vector=np.array([true_bearing]),
                timestamp=timestamp
            )
            measurement = detection_measurement_model.function(
                true_bearing_state, noise=True
            )

            detection = Detection(
                state_vector=measurement,
                timestamp=timestamp,
                measurement_model=ss_measurement_model
            )
            detections_at_time.append(detection)

            # Generate ambiguous detection
            if SS_DET_PARAMS["include_ambiguity"]:
                platform_velocity = platform_state.host.state.state_vector[[1, 3]]
                platform_heading = np.arctan2(
                    platform_velocity[1], platform_velocity[0]
                )

                # Get array orientation from actual array geometry (not velocity)
                # The array axis points from tail to head of the array
                array_positions = platform_state.array.state_vector
                array_head = array_positions[:2, -1]  # Last sensor (head)
                array_tail = array_positions[:2, 0]   # First sensor (tail)
                array_axis = array_head - array_tail
                array_heading = np.arctan2(array_axis[1], array_axis[0])

                ambiguous_bearing = generate_ambiguous_bearing(
                    true_bearing, array_heading
                )

                ambiguous_bearing_state = GroundTruthState(
                    state_vector=np.array([ambiguous_bearing]),
                    timestamp=timestamp
                )
                ambiguous_measurement = detection_measurement_model.function(
                    ambiguous_bearing_state, noise=True
                )

                ambiguous_detection = Detection(
                    state_vector=ambiguous_measurement,
                    timestamp=timestamp,
                    measurement_model=ss_measurement_model
                )
                detections_at_time.append(ambiguous_detection)

    # Add clutter
    num_clutter = np.random.poisson(clutter_spatial_density)
    for _ in range(num_clutter):
        clutter_bearing = uniform.rvs(loc=-np.pi, scale=2*np.pi)
        clutter_detection = Detection(
            state_vector=np.array([[clutter_bearing]]),
            timestamp=timestamp,
            measurement_model=ss_measurement_model
        )
        detections_at_time.append(clutter_detection)

    stone_soup_detections.append((timestamp, detections_at_time))

ss_detections_for_plotter = [det_set for _, det_set in stone_soup_detections]

In [ ]:
fig = make_subplots(
    rows=1, cols=2, shared_xaxes=True, shared_yaxes=True, horizontal_spacing=0.06,
    subplot_titles=["(a)", "(b)"]
)

fig.add_trace(
    go.Scatter(
        x=det_x,
        y=det_y,
        mode="markers",
        name="Detection",
        showlegend=False,
        marker=dict(
            size=6,
            line=dict(width=0.5),
            color="white",
            opacity=0.8,
        ),
        hovertemplate="Bearing: %{x:.1f}°<br>Time: %{y|%H:%M:%S}<extra></extra>",
    ),
    row=1, col=1
)

ss_det_x = []
ss_det_y = []

for t, detection_set in stone_soup_detections:
    for detection in detection_set:
        ss_det_x.append(np.rad2deg(detection.state_vector[0]))
        ss_det_y.append(t)

fig.add_trace(
    go.Scatter(
        x=ss_det_x,
        y=ss_det_y,
        mode="markers",
        name="Detection",
        showlegend=True,
        marker=dict(
            size=6,
            line=dict(width=0.5),
            color="white",
            opacity=0.8,
        ),
        hovertemplate="Bearing: %{x:.1f}°<br>Time: %{y|%H:%M:%S}<extra></extra>",
    ),
    row=1, col=2
)

fig.update_xaxes(
    range=[-180, 180],
    tickmode="linear",
    tick0=-180,
    dtick=60,
    tickangle=-45,
    tickfont=dict(size=14),
    ticks="outside",
    tickcolor="rgba(160, 160, 160, 1.0)",
    showgrid=True,
    gridcolor="rgba(200, 200, 200, 0.5)",
    showline=True,
    linecolor="rgba(160, 160, 160, 1.0)",
)
fig.add_annotation(
    text="Bearing (°)",
    xref="paper", yref="paper",
    x=0.5,
    y=-0.18,
    showarrow=False,
    font=dict(family="Times New Roman", size=18, color="black")
)

fig.update_yaxes(
    range=[timesteps[-1], timesteps[0]],
    tickformat="%H:%M",
    tickfont=dict(size=16),
    ticks="outside",
    tickcolor="rgba(160, 160, 160, 1.0)",
    showgrid=True,
    gridcolor="rgba(200, 200, 200, 0.5)",
    showline=True,
    linewidth=1,
    linecolor="rgba(160, 160, 160, 1.0)",
    autorange=False,
    title="Time (HH:MM)"
)
fig.update_yaxes(title="", ticks="", showticklabels=False, col=2)

fig.update_layout(
    width=600,
    height=600,
    margin=dict(b=100),
    font=dict(family="Times New Roman", size=16, color="black"),
    showlegend=False,
    legend=dict(
        x=1.0,
        y=1.02,
        xanchor="right",
        yanchor="bottom",
        bgcolor="rgba(255,255,255,0.0)",
        borderwidth=0,
        orientation="h"
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
)

fig.show()

fig.write_image("figs/mt_plugin_vs_ss.pdf", scale=1, width=600, height=600)